In [4]:
import pandas as pd
import os
import sys

# --- Configuration ---
# Update this path to your MIMIC-Extract HDF5 file.
HDF_FILE_PATH = '../data/raw/all_hourly_data.h5'

def inspect_hdf5_file_robustly(filepath):
    """
    Reads and displays the head of each table in a MIMIC-Extract HDF5 file,
    handling potential format incompatibilities.

    Args:
        filepath (str): The full path to the .h5 file.
    """
    if not os.path.exists(filepath):
        print(f"--- CRITICAL ERROR: File not found ---")
        print(f"The path '{filepath}' does not exist. Please check your configuration.")
        sys.exit(1) # Exit with an error code

    print(f"--- Begin HDF5 File Inspection ---")
    print(f"Target file: {filepath}\n")
    print(f"Pandas version being used: {pd.__version__}\n")

    try:
        # Use HDFStore in read-only mode ('r'). This is safer.
        with pd.HDFStore(filepath, 'r') as store:
            # Discover the keys (table names) within the file.
            all_keys = store.keys()
            if not all_keys:
                print("--- WARNING: The HDF5 file is empty or unreadable. ---")
                return

            print(f"Discovered the following tables (keys) in the file: {all_keys}\n")

            for key in all_keys:
                print(f"==========================================================")
                print(f"Table: '{key}'")
                print(f"==========================================================")

                try:
                    # METHOD 1 (Ideal): Memory-efficient read of the first 5 rows.
                    # This uses the HDF5 file's query engine (where clause) and is
                    # the correct way to handle large tables.
                    df_head = store.select(key, start=0, stop=5)
                    print(df_head)

                except (TypeError, IndexError, ValueError) as e:
                    # RATIONALE: These errors are symptomatic of a format mismatch.
                    # The query engine in modern pandas cannot interpret the file's
                    # index or table format correctly to perform a ranged select.
                    print(f"--> INFO: Standard query failed for '{key}'. This is expected for older HDF5 formats.")
                    print(f"--> Error was: {type(e).__name__}: {e}")
                    print("--> Attempting alternative full-select method (still efficient if it works)...")

                    try:
                        # METHOD 2 (Fallback): Select the entire table (efficiently) and then
                        # take the head in-memory. This often works when start/stop fails.
                        df_full = store.select(key)
                        df_head = df_full.head(5)
                        print(df_head)
                    except Exception as fallback_e:
                        print(f"--- FAILURE ---")
                        print(f"All efficient read methods have failed for table '{key}'.")
                        print(f"This indicates a severe incompatibility or file corruption.")
                        print(f"Fallback Error: {fallback_e}")

                except Exception as e:
                    print(f"--- UNEXPECTED FAILURE ---")
                    print(f"A non-standard error occurred while reading '{key}': {e}")

                print("\n")

    except Exception as e:
        print(f"--- A CRITICAL Error Occurred Opening the HDF5 Store ---")
        print(f"Could not process the HDF5 file itself. It may be corrupted or not a valid HDF5 file.")
        print(f"Error: {e}")


In [5]:

inspect_hdf5_file_robustly(HDF_FILE_PATH)

--- Begin HDF5 File Inspection ---
Target file: ../data/raw/all_hourly_data.h5

Pandas version being used: 0.24.2

Discovered the following tables (keys) in the file: ['/codes', '/interventions', '/patients', '/vitals_labs', '/vitals_labs_mean', '/patients/meta/values_block_6/meta', '/patients/meta/values_block_5/meta', '/patients/meta/values_block_4/meta', '/patients/meta/values_block_0/meta']

Table: '/codes'
--> INFO: Standard query failed for '/codes'. This is expected for older HDF5 formats.
--> Error was: IndexError: index -72 is out of bounds for axis 0 with size 5
--> Attempting alternative full-select method (still efficient if it works)...
                                                                               icd9_codes
subject_id hadm_id icustay_id hours_in                                                   
3          145834  211552     0         [0389, 78559, 5849, 4275, 41071, 4280, 6826, 4...
                              1         [0389, 78559, 5849, 4275, 41071,